In [1]:
import sys
import warnings
import os
import random
import numpy
from numpy import mean
from numpy import std
from sklearn.datasets import make_classification
from sklearn.model_selection import KFold
from sklearn.model_selection import RepeatedKFold
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import normalize
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.dummy import DummyClassifier

In [164]:
#Data parameters

data_file = "Data/CNM_004.csv"
data_file_header_lines = 2
data_file_fields_separator = ","
# Argument: Block number starting from 0 (number)
classes_file_template = "Data/Day2_%d.tsv"
classes_files_num = 5
classes_file_header_lines = 1
classes_file_fields_separator = "\t"
classes_file_timestamp_shift = 5.05
classes_output_file = "Data/Classes.csv"
samples_output_file = "Data/Samples.csv"
chunks_output_file = "Data/Chunks.csv"
chunk_length = 0.1 #length of features in seconds
chunk_generation_backward = False
random_mean = 50 #random gaussian
random_std = 30 #random gaussian
# Summary function selector, values are "mean", "mode", "min", "max", "random" (for random gaussian feature values)
evaluate_summary = "mean" 
# One or more samples sets (<class_id>_<block_#>) per evaluating class
# example ["4_2", "4_1"] is set for class 1 and ["3_2", "3_1"] is set for class 2
#evaluate_class = [["4_2", "4_1"], ["3_2", "3_1"]]
#evaluate_class = [["4_4", "4_3"], ["3_4", "3_3"]]

evaluate_class = [["5_3"], ["6_3"]]
evaluate_class = [["4_3"], ["3_3"]]

#evaluate_class = [["4_2"], ["3_2"]]
#evaluate_class = [["4_4"], ["3_4"]] -- why is this throwing an error?
# True for random class labels
evaluate_random_class_labels = False #Classify random gaussian labels
evaluate_scoring = "accuracy"
#features_selector = ["C45"] # select featured to use ["all"] - all, ["accepted"] - only accepted, ["rejected"] - only rejected, ["C00", "C14", "C20"] - use only listed features, ["try"] - try all features one-by-one (executes evaluation several times)
#features_selector = ["C09"] # select featured to use ["all"] - all, ["accepted"] - only accepted, ["rejected"] - only rejected, ["C00", "C14", "C20"] - use only listed features, ["try"] - try all features one-by-one (executes evaluation several times)
features_selector = ["all"] # select featured to use ["all"] - all, ["accepted"] - only accepted, ["rejected"] - only rejected, ["C00", "C14", "C20"] - use only listed features, ["try"] - try all features one-by-one (executes evaluation several times)
features_selector = ["C82"] # select featured to use ["all"] - all, ["accepted"] - only accepted, ["rejected"] - only rejected, ["C00", "C14", "C20"] - use only listed features, ["try"] - try all features one-by-one (executes evaluation several times)

normalize_features = StandardScaler() # Normalizer of None to disamle feaature normalization
evaluate_models = [
    #{"model":DummyClassifier(strategy='most_frequent', random_state = 1), "name": "DummyMostFrequent"},
    {"model":LinearSVC(random_state = 1), "name": "LinearSVC"}]
evaluate_num_subsamplings = 30
evaluate_num_repetitions = 50

In [165]:
if not sys.warnoptions:
    warnings.simplefilter("ignore")
    os.environ["PYTHONWARNINGS"] = "ignore"

In [166]:
def read_file(file, header_lines, separator):
    headers = []
    data = {}
    with open(file, "r") as f_file:
        line_num = 0
        for line in f_file:
            line_num += 1
            header = False
            if line_num <= header_lines:
                header = True
            line = line.replace(separator + " ", separator).strip(" \n\r")
            if len(line) > 0:
                row = line.split(separator)
                values = []
                values_good = True
                if header:
                     headers.append(row)
                else:
                    timestamp_str = row[0]
                    timestamp = -1
                    try:
                        timestamp = float(timestamp_str)
                    except:
                        print("Error timestamp value '%s' in line #%d" % (timestamp_str, line_num))
                    if timestamp >= 0:
                        payload = row[1:]
                        data[timestamp_str] = payload
    return {"headers": headers, "data": data}

In [167]:
__data = read_file(data_file, data_file_header_lines, data_file_fields_separator)
data = __data["data"]
headers = __data["headers"][0][1:]
flags = __data["headers"][1][1:]
headers_flags = []
if len(headers) == len(flags):
    for i in range(len(headers)):
        headers_flags.append(headers[i] + ":" + flags[i])
else:
    print("Error matching features and their flags")
features_num = len(headers_flags)
print("Data points: %d" % (len(data)))
print("Features: %d" % (len(headers)))
data_block_offsets = []
prev_timestamp = None
cur_idx = 0
for timestamp in data:
    if prev_timestamp == None or float(timestamp) - float(prev_timestamp) > 1:
        data_block_offsets.append(cur_idx)
    cur_idx += 1
    prev_timestamp = timestamp
print("Data blocks: %d" % (len(data_block_offsets)))

Data points: 30025
Features: 92
Data blocks: 5


In [168]:
samples = {}
f_file = open(classes_output_file, "w")
f_chunks = open(chunks_output_file, "w")
f_file.write("class_id, block_#, instance_#, timestamp, " + (", ".join(headers_flags)) + "\n")
f_chunks.write("class_id, block_#, instance_#, chunk_id, timestamp, " + (", ".join(headers_flags)) + "\n")
for classes_file_idx in range(classes_files_num):
    classes_file = classes_file_template % (classes_file_idx)
    __classes = read_file(classes_file, classes_file_header_lines, classes_file_fields_separator)
    classes = __classes["data"]
    class_ids = __classes["headers"][0][1:]
    block = classes_file_idx
    print("Block #%d has %d annotations with %d classes: %s" % (block, len(classes), len(class_ids), str(class_ids)))
    for class_timestamp in classes:
        class_flags_cnt = 0
        for class_flag in classes[class_timestamp]:
            if class_flag != "0":
                class_flags_cnt += 1
        if class_flags_cnt > 1:
            print("Error in annotations. %d class flags (>1) is defined for timestamp %s" % (class_flags_cnt, class_timestamp))
            classes[class_timestamp] = ["0" for i in range(len(classes[class_timestamp]))]
    classes_running = [False for col in range(len(class_ids))]
    classes_instance = [-1 for col in range(len(class_ids))]
    data_offset = data_block_offsets[block]
    data_size = (data_block_offsets[block + 1] if block < len(data_block_offsets) - 1 else len(data)) - data_block_offsets[block]
    classes_timestamps = list(classes.keys())
    cur_classes_idx = 0
    data_block_timestamps = list(data.keys())[data_offset:data_offset+data_size]
    data_block_timestamp_shift = float(data_block_timestamps[0])
    #print("%f, %f" % (data_block_timestamp_shift, classes_file_timestamp_shift))
    instance_timestamps = []
    data_idx = 0
    for data_timestamp in data_block_timestamps:
        while cur_classes_idx < len(classes_timestamps) and (float(data_timestamp) - data_block_timestamp_shift > float(classes_timestamps[cur_classes_idx]) - classes_file_timestamp_shift or float(classes_timestamps[cur_classes_idx]) - classes_file_timestamp_shift < 0):
            cur_classes_idx += 1
        if cur_classes_idx >= len(classes_timestamps):
            break
        #print("%f => %f [%d]" % (data_timestamp - data_block_timestamp_shift, classes_timestamps[cur_classes_idx] - classes_file_timestamp_shift, cur_classes_idx))
        for class_idx in range(len(class_ids)):
            if classes[classes_timestamps[cur_classes_idx]][class_idx] == "0":
                if classes_running[class_idx]:
                    classes_running[class_idx] = False
                    instance_interval = float(instance_timestamps[-1]) - float(instance_timestamps[0])
                    if instance_interval >= chunk_length:
                        chunks = []
                        if chunk_generation_backward:
                            chunk = []
                            for timestamp in instance_timestamps[::-1]:
                                if len(chunk) > 0:
                                    if float(chunk[0]) - float(timestamp) > chunk_length or (float(chunk[0]) - float(timestamp) >= chunk_length and timestamp == instance_timestamps[-1]):
                                        chunk.reverse()
                                        #print("Reverse chunk: " + str(chunk))
                                        chunks.append(chunk)
                                        chunk = []
                                chunk.append(timestamp)
                        else:
                            chunk = []
                            for timestamp in instance_timestamps:
                                if len(chunk) > 0:
                                    if float(timestamp) - float(chunk[0]) > chunk_length or (float(timestamp) - float(chunk[0]) >= chunk_length and timestamp == instance_timestamps[-1]):
                                        #print("Straigh chunk: " + str(chunk))
                                        chunks.append(chunk)
                                        chunk = []
                                chunk.append(timestamp)
                        chunk_id = -1
                        for chunk in chunks:
                            chunk_id += 1
                            chunk_values = [[] for i in range(features_num)]
                            chunk_mean = [0 for i in range(features_num)]
                            chunk_min = [None for i in range(features_num)]
                            chunk_max = [None for i in range(features_num)]
                            chunk_mode = [None for i in range(features_num)]
                            chunk_mode_arr = [{} for i in range(features_num)]
                            for timestamp in chunk:
                                element = data[timestamp]
                                f_chunks.write(("%s, %d, %d, %d, %s, " % (class_ids[class_idx], block, classes_instance[class_idx], chunk_id, timestamp)) + (", ".join(element)) + "\n")
                                for i in range(features_num):
                                    val = float(element[i])
                                    chunk_values[i].append(val)
                                    chunk_mean[i] += val
                                    if chunk_min[i] == None or chunk_min[i] > val:
                                        chunk_min[i] = val
                                    if chunk_max[i] == None or chunk_max[i] < val:
                                        chunk_max[i] = val
                                    if val not in chunk_mode_arr[i]:
                                        chunk_mode_arr[i][val] = 0
                                    chunk_mode_arr[i][val] += 1
                            features = {"mean": [], "mode": [], "min": [], "max": [], "random": []}
                            random_features = numpy.random.normal(random_mean, random_std, features_num).tolist()
                            for i in range(features_num):
                                chunk_mean[i] /= len(chunk)
                                chunk_mode_arr[i] = dict(sorted(chunk_mode_arr[i].items(), key=lambda item: item[1], reverse=True))
                                chunk_mode[i] = next(iter(chunk_mode_arr[i]))
                                #print("[%d] mean = %f, mode = %f, min = %f, max = %f" % (i, chunk_mean[i], chunk_mode[i], chunk_min[i], chunk_max[i]))
                                features["mean"].append(chunk_mean[i])
                                features["mode"].append(chunk_mode[i])
                                features["min"].append(chunk_min[i])
                                features["max"].append(chunk_max[i])
                                features["random"].append(random_features[i])
                            sample_id = class_ids[class_idx] + "_" + str(block)
                            if not sample_id in samples:
                                samples[sample_id] = []
                            samples[sample_id].append(features)
                    else:
                        print("Instance %s - %s = %f ms is too short (<100ms). Class %s, block #%d, instance #%d. Annotation at %s" % (instance_timestamps[0], instance_timestamps[-1], instance_interval, class_ids[class_idx], block, classes_instance[class_idx], classes_timestamps[cur_classes_idx]))
                    instance_timestamps = []
        for class_idx in range(len(class_ids)):
            if classes[classes_timestamps[cur_classes_idx]][class_idx] != "0":
                if not classes_running[class_idx]:
                    classes_instance[class_idx] += 1
                    classes_running[class_idx] = True
                f_file.write(("%s, %d, %d, %s, " % (class_ids[class_idx], block, classes_instance[class_idx], data_timestamp)) + (", ".join(data[data_timestamp])) + "\n")
                instance_timestamps.append(data_timestamp)
f_file.close()
f_chunks.close()
for sample_id in samples:
    print("Sample group %s has %d samples" % (sample_id, len(samples[sample_id])))

Block #0 has 6100 annotations with 2 classes: ['1', '2']
Instance 128.649575 - 128.699536 = 0.049961 ms is too short (<100ms). Class 2, block #0, instance #5. Annotation at 133.8
Block #1 has 6101 annotations with 5 classes: ['3', '4', '5', '6', '7']
Block #2 has 6100 annotations with 4 classes: ['3', '4', '5', '6']
Block #3 has 6100 annotations with 4 classes: ['3', '4', '5', '6']
Block #4 has 6101 annotations with 4 classes: ['3', '4', '5', '6']
Sample group 1_0 has 105 samples
Sample group 2_0 has 234 samples
Sample group 4_1 has 107 samples
Sample group 5_1 has 66 samples
Sample group 3_1 has 14 samples
Sample group 6_1 has 23 samples
Sample group 7_1 has 9 samples
Sample group 5_2 has 147 samples
Sample group 3_2 has 27 samples
Sample group 6_2 has 44 samples
Sample group 4_2 has 117 samples
Sample group 4_3 has 267 samples
Sample group 5_3 has 59 samples
Sample group 6_3 has 35 samples
Sample group 3_3 has 11 samples
Sample group 4_4 has 264 samples
Sample group 5_4 has 4 samples

In [169]:
selected_features = features_selector
if selected_features[0] != "all" and selected_features[0] != "accepted" and selected_features[0] != "rejected":
    if selected_features[0] == "try":
        selected_features = headers
        print("Features selected for independent evaluation: %s" % (",".join(selected_features)))
    else:
        selected_features = [",".join(selected_features)]
        print("Features selected for group evaluation: %s" % selected_features)
else:
    print("Multiple features for group evaluation: %s" % (selected_features[0]))
for selected_feature in selected_features:
    X_class = [[], []]
    y_class = [[], []]
    class_samples_cnt = [0, 0]
    random.seed(1)
    with open(samples_output_file, "w") as f_samples:
        f_samples.write("Class_ID, Source_Sample_ID, %s" % (", ".join(headers_flags)))
        for evaluate_class_idx in range(2):
            for sample_id in evaluate_class[evaluate_class_idx]:
                if sample_id in samples:
                    _X = [samples[sample_id][i][evaluate_summary] for i in range(len(samples[sample_id]))]
                    _y = [(random.randint(1,2) if evaluate_random_class_labels else (evaluate_class_idx + 1)) for i in range(len(samples[sample_id]))]
                    labels = numpy.array(_y)
                    class_samples_cnt[evaluate_class_idx] += numpy.count_nonzero(labels == evaluate_class_idx + 1)
                    for i in range(len(_y)):
                        X_class[_y[i] - 1].append([])
                        for feature_idx in range(len(_X[i])):
                            if selected_feature == "all" or flags[feature_idx] == selected_feature or headers[feature_idx] in selected_feature:
                                X_class[_y[i] - 1][-1].append(_X[i][feature_idx])
                        y_class[_y[i] - 1].append(_y[i])
                    for j in range(len(_y)):
                        f_samples.write("%d, %s, %s\n" % (_y[j], sample_id, ", ".join([str(x) for x in _X[j]])))
                else:
                    print("Error: %s sample set not found in class %s" % (sample_id, evaluate_class_idx + 1))
    #for evaluate_class_idx in range(2):
    #    print("Class %d:" % (evaluate_class_idx + 1))
    #    print("   Samples: %d" % (class_samples_cnt[evaluate_class_idx]))
    #    print("   X vector size: %d" % (len(X_class[evaluate_class_idx])))
    #    print("   X vector element size: %d" % (len(X_class[evaluate_class_idx][0])))
    #    print("   y vector size: %d" % (len(y_class[evaluate_class_idx])))
    scores = {}
    for evaluate_model in evaluate_models:
        scores[evaluate_model["name"]] = []
    for repetition_idx in range(evaluate_num_repetitions):
        for subsample_idx in range(evaluate_num_subsamplings):
            min_samples_num = min(class_samples_cnt)
            X = []
            y = []
            random.seed(repetition_idx * evaluate_num_subsamplings + subsample_idx)
            for evaluate_class_idx in range(2):
                idxs = [i for i in range(len(y_class[evaluate_class_idx]))]
                random.shuffle(idxs)
                for i in range(min_samples_num):
                    X.append(X_class[evaluate_class_idx][idxs[i]])
                    y.append(y_class[evaluate_class_idx][idxs[i]])
            if normalize_features != None:
                X = normalize_features.fit_transform(X)
            #print("X vector of %d elements" % (len(X)))
            #print("y vector of %d elements" % (len(y)))
            #print("Classes distribution: %s" % (str(dict(zip(*numpy.unique(y, return_counts=True))))))
            for evaluate_model in evaluate_models:
                cv = StratifiedKFold(n_splits = 10)
                scores[evaluate_model["name"]] += list(cross_val_score(evaluate_model["model"], X, y, scoring = evaluate_scoring, cv = cv, n_jobs = -1))
                #print("Detailed accuracy scores: %s" % (str(scores)))
    for evaluate_model in evaluate_models:
        _scores = sorted(scores[evaluate_model["name"]], key = lambda x:float(x), reverse = True)
        print("Feature(s): %s. Model: %s. Accuracy: %.3f (%.3f)" % (selected_feature, evaluate_model["name"], mean(scores[evaluate_model["name"]]), std(scores[evaluate_model["name"]])))
        #print("   10 best runs: %s" % (",".join([str(x) for x in _scores[:10]])))
        #print("   10 worst runs: %s" % (",".join([str(x) for x in _scores[-10:]])))


Features selected for group evaluation: ['C82']
Feature(s): C82. Model: LinearSVC. Accuracy: 0.901 (0.192)
